# Seq2Seq 모델 Q&A + Chatbot 만들기

1. qna 데이터셋을 찾아서 처리해서 준비한다.( 전처리 전반)
2. 인코더디코더seq2seq(인코더+디코더) 모델 만들기
3. 1에서 준비한 데이터로 2에서 만든 모델을 학습
4. 챗봇을 만든다(모델 추론 +while문)

In [2]:
# %pip install datasets

### BCCard-Finance-Kor-QnA
- 제작기관: BC Card
- 도메인: 금융 (카드, 결제, 금융서비스)
- 데이터 용량: 31250
- 목적: 금융 지식 기반 챗봇 및 금융 도메인 LLM 학습
- 특징: 금융 서비스 중심 질문(카드 결제, 발급, 금융 서비스, 제도, 연체 이자 등)


https://huggingface.co/datasets/BCCard/BCCard-Finance-Kor-QnA?utm_source=chatgpt.com&library=datasets

In [1]:
from datasets import load_dataset

ds = load_dataset("BCCard/BCCard-Finance-Kor-QnA")

In [2]:
ds

DatasetDict({
    train: Dataset({
        features: ['instruction', 'output'],
        num_rows: 31250
    })
})

In [3]:
ds["train"][0]

{'instruction': '비씨카드는 어떤 회사인가요?',
 'output': "비씨카드(BC카드)는 1982년에 설립된 대한민국의 대표적인 카드사로, 주로 결제 프로세싱 서비스를 제공하는 회사입니다. 설립 초기에는 '은행신용카드협회'라는 이름으로 시작했으며, BC는 'Bank Credit'의 약자입니다. 이 협회는 조흥은행, 상업은행, 서울은행, 제일은행, 한일은행 등 주요 은행들이 공동으로 설립하여, 카드 결제 시스템을 효율적으로 관리하고, 국내 카드 결제 인프라를 구축하는 역할을 했습니다.\n\n비씨카드는 이후 독립적인 신용카드 회사로 성장하며, 카드 발급, 결제 처리, 가맹점 관리 등의 다양한 금융 서비스를 제공하게 되었습니다. 현재 비씨카드는 KT가 최대 주주로, KT의 빅데이터 및 AI 기술을 활용해 디지털 금융 서비스를 강화하고 있습니다. 비씨카드는 결제 프로세싱 시장에서 오랜 경험과 노하우를 바탕으로, 국내외에서 다양한 금융 서비스를 제공하며 카드사 중에서도 중요한 위치를 차지하고 있습니다.\n\n비씨카드는 단순히 카드 발급과 결제에 그치지 않고, 디지털 결제 플랫폼 '페이북'을 운영하며, 중금리 대출, 마이데이터 사업 등으로 서비스 범위를 확장하고 있습니다. 이러한 디지털 혁신은 비씨카드가 빅테크 기업들과의 경쟁에서도 경쟁력을 유지하기 위한 핵심 전략입니다. \n\n이처럼 비씨카드는 전통적인 카드사로서의 역할을 유지하면서도, 디지털 금융의 선두주자로 자리매김하기 위해 지속적으로 변화를 추구하고 있는 기업입니다."}

In [4]:
import re
import pandas as pd
import numpy as np
from konlpy.tag import Okt
import torch
import torch.nn as nn
import torch.functional as F
from torch.utils.data import Dataset, DataLoader

In [22]:
# 전역변수
BATCH_SIZE = 64
MAX_VOCAB_SIZE = 30000
EMBEDDING_DIM = 100
LATENT_DIM = 512

In [ ]:

okt = Okt()


with open('./ko_stopwords_cleaned.txt', 'r', encoding='utf-8') as f:
    stopword_list = [line.strip() for line in f if line.strip()]

# 토큰화 함수
def tokenizer(text):
    # 입력 문장을 형태소 단위로 분리
    tokens = okt.morphs(text)
    return tokens

# 불용어 제거 함수
def remove_stopwords(tokens):
    # 미리 불러온 stopword_list에 포함되지 않은 토큰만 남긴다.
    return [token for token in tokens if token not in stopword_list]

# 토큰 리스트를 다시 문장으로 합치는 함수
def detokenize(tokens):
    # 토큰 사이를 공백으로 연결
    sentence = " ".join(tokens)

    # 구두점 앞 공백 제거
    sentence = re.sub(r"\s+([?.!,~])", r"\1", sentence)

    # 괄호 주변 공백 정리
    sentence = re.sub(r"\(\s+", "(", sentence)
    sentence = re.sub(r"\s+\)", ")", sentence)

    # 여러 공백을 하나로 축소
    sentence = re.sub(r"\s+", " ", sentence).strip()

    return sentence

# 전체 데이터 전처리 함수
def preprocess_data(data):
    preprocessed_data = []

    for item in data:
        # 원본 질문과 답변 추출
        question = item['instruction']
        answer = item['output']

        # 1. 질문/답변 토큰화
        question_tokens = tokenizer(question)
        answer_tokens = tokenizer(answer)

        # 2. 불용어 제거
        question_tokens = remove_stopwords(question_tokens)
        answer_tokens = remove_stopwords(answer_tokens)

        # 3. 토큰 리스트를 다시 문장으로 변환
        question_sentence = detokenize(question_tokens)
        answer_sentence = detokenize(answer_tokens)

        # 4. 전처리된 결과 저장
        preprocessed_data.append({
            'question': question_sentence,
            'answer': answer_sentence
        })

    return preprocessed_data

# 전처리 실행
preprocessed_ds = preprocess_data(ds['train'])

# 결과 확인
print(preprocessed_ds[0])

In [5]:
# 리스트 데이터 저장
# preprocessed_df = pd.DataFrame(preprocessed_ds)
# preprocessed_df.to_csv('./preprocessed_data.csv', index=False, encoding='utf-8-sig')
# 전처리된 데이터 불러오기
preprocessed_df = pd.read_csv('./preprocessed_data.csv', encoding='utf-8-sig')
preprocessed_ds = preprocessed_df.to_dict(orient='records')


FileNotFoundError: [Errno 2] No such file or directory: './chat_data.csv'

In [8]:
# vocab 만들기
from collections import Counter

special_tokens = ['<PAD>', '<SOS>', '<EOS>', '<UNK>']

counter = Counter()

for item in preprocessed_ds:
    question_tokens = item['question'].split()
    answer_tokens = item['answer'].split()
    
    counter.update(question_tokens)
    counter.update(answer_tokens)

vocab = special_tokens + list(counter.keys())
word2idx = {word: idx for idx, word in enumerate(vocab)}
idx2word = {idx: word for idx, word in word2idx.items()}

print(word2idx['비씨'])
print(word2idx['카드'])

948
17


In [14]:
# 문장을 정수 시퀀스
def sentence_to_sequence(tokens, word2idx):
    return [word2idx.get(token, word2idx['<UNK>']) for token in tokens]

tokenized_data = []

for sample in preprocessed_ds:
    
    question_tokens = sample['question'].split()
    answer_tokens = sample['answer'].split()

    # 질문 문장
    q_ids = sentence_to_sequence(question_tokens, word2idx)

    # 답변 문장
    a_ids = [word2idx['<SOS>']] + sentence_to_sequence(answer_tokens, word2idx) + [word2idx['<EOS>']]

    tokenized_data.append({
        'question_ids': q_ids,
        'answer_ids': a_ids
    })

print(tokenized_data[0])

{'question_ids': [4, 5, 6, 7], 'answer_ids': [1, 4, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 11, 26, 27, 28, 29, 30, 31, 28, 32, 33, 34, 35, 36, 28, 37, 38, 28, 39, 31, 40, 41, 42, 43, 42, 44, 42, 45, 46, 42, 47, 46, 29, 48, 29, 49, 11, 50, 17, 20, 51, 52, 15, 53, 54, 55, 17, 20, 56, 57, 24, 58, 59, 4, 60, 61, 15, 16, 30, 6, 62, 17, 63, 20, 64, 65, 66, 53, 67, 68, 22, 23, 69, 70, 71, 4, 72, 73, 74, 72, 75, 76, 77, 78, 79, 80, 68, 22, 81, 82, 4, 20, 21, 83, 84, 85, 86, 87, 88, 67, 68, 22, 23, 89, 17, 90, 91, 92, 93, 94, 95, 96, 82, 4, 97, 17, 98, 20, 99, 100, 80, 20, 101, 28, 102, 103, 28, 104, 105, 91, 106, 107, 108, 109, 110, 22, 111, 112, 96, 82, 113, 80, 114, 4, 115, 116, 117, 118, 119, 92, 120, 121, 122, 123, 47, 124, 125, 126, 4, 127, 15, 16, 17, 90, 128, 58, 121, 129, 80, 68, 130, 131, 132, 133, 134, 135, 15, 136, 137, 96, 138, 139, 2]}


In [7]:
# padding
max_q_len = max(len(sample['question_ids']) for sample in tokenized_data)
max_a_len = max(len(sample['answer_ids']) for sample in tokenized_data)

def pad_sequence(seq, max_len, pad_value):
    return seq + [pad_value] * (max_len - len(seq))

pad_idx = word2idx['<PAD>']
for sample in tokenized_data:
    sample['question_ids'] = pad_sequence(sample['question_ids'], max_q_len, pad_idx)
    sample['answer_ids'] = pad_sequence(sample['answer_ids'], max_a_len, pad_idx)
    
print((tokenized_data[0]))

{'question_ids': [4, 5, 6, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'answer_ids': [1, 4, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 11, 26, 27, 28, 29, 30, 31, 28, 32, 33, 34, 35, 36, 28, 37, 38, 28, 39, 31, 40, 41, 42, 43, 42, 44, 42, 45, 46, 42, 47, 46, 29, 48, 29, 49, 11, 50, 17, 20, 51, 52, 15, 53, 54, 55, 17, 20, 56, 57, 24, 58, 59, 4, 60, 61, 15, 16, 30, 6,

219

In [ ]:
# seq2seq 학습용구조
# seq2seq_data = []

# for sample in tokenized_data:
#     encoder_input = sample['question_ids']
#     decoder_input = sample['answer_ids'][:-1]
#     decoder_target = sample['answer_ids'][1:]
    
#     seq2seq_data.append({
#         'encoder_input': encoder_input,
#         'decoder_input': decoder_input,
#         'decoder_target': decoder_target
#     })

# print(seq2seq_data[0])

In [29]:
# dataset/ dataloader 만들기
class ChatDataset(Dataset):
    def __init__(self, data):
        self.data = data
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        sample = self.data[idx]
        # 질문은 encoder 입력
        encoder_input = sample['question_ids']

        # 답변은 decoder 입력 / target 으로 한 칸씩 나눔
        answer_ids = sample['answer_ids']
        decoder_input = answer_ids[:-1]   # 마지막 <EOS> 제외
        decoder_target = answer_ids[1:]
        return {
            'encoder_input': torch.tensor(encoder_input, dtype=torch.long),
            'decoder_input': torch.tensor(decoder_input, dtype=torch.long),
            'decoder_target': torch.tensor(decoder_target, dtype=torch.long)
        }
chat_dataset = ChatDataset(tokenized_data)
dataloader = DataLoader(chat_dataset, batch_size=32, shuffle=True)

In [30]:
# Encoder 모델 구현
class Encoder(nn.Module):
  def __init__(self, vocab_size, embedding_dim, hidden_dim, pad_idx, embedding_matrix=None):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
    if embedding_matrix is not None:
      self.embedding.weight.data.copy_(torch.from_numpy(embedding_matrix))
    self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)

  def forward(self, X):
    X = self.embedding(X)
    output, (h_s, c_s) = self.lstm(X)
    return h_s, c_s

In [31]:
# Decoder 모델 구현
class Decoder(nn.Module):
  def __init__(self, vocab_size, embedding_dim, hidden_dim, pad_idx):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
    self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
    self.fc = nn.Linear(hidden_dim, vocab_size)

  def forward(self, X, hidden, cell):
    X = self.embedding(X)
    output, (h_s, c_s) = self.lstm(X, (hidden, cell))
    logits = self.fc(output)
    return logits, (h_s, c_s)

In [32]:
# Seq2Seq 모델 구현
class Seq2Seq(nn.Module):
  def __init__(self, encoder, decoder):
    super().__init__()
    self.encoder = encoder
    self.decoder = decoder

  def forward(self, source, target):
    h_s, c_s = self.encoder(source)
    # h_s, c_s를 튜플이 아닌 개별 인자로 전달합니다.
    output, (h_s, c_s) = self.decoder(target, h_s, c_s)
    return output

In [33]:
vocab_size = len(word2idx)

encoder = Encoder(
    vocab_size,
    EMBEDDING_DIM,
    LATENT_DIM,
    pad_idx=word2idx['<PAD>'],
    embedding_matrix=None
)

decoder = Decoder(
    vocab_size,
    EMBEDDING_DIM,
    LATENT_DIM,
    pad_idx=word2idx['<PAD>']
)

model = Seq2Seq(encoder, decoder)
model

Seq2Seq(
  (encoder): Encoder(
    (embedding): Embedding(35705, 100, padding_idx=0)
    (lstm): LSTM(100, 512, batch_first=True)
  )
  (decoder): Decoder(
    (embedding): Embedding(35705, 100, padding_idx=0)
    (lstm): LSTM(100, 512, batch_first=True)
    (fc): Linear(in_features=512, out_features=35705, bias=True)
  )
)

In [34]:
train_loader = DataLoader(chat_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(chat_dataset, batch_size=BATCH_SIZE)

In [35]:
criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

epochs = 5

train_losses, train_accs, val_losses, val_accs = [], [], [], []

for epoch in range(epochs):

    model.train()
    train_loss, train_correct, train_tokens = 0, 0, 0

    for batch in train_loader:

        enc_inputs = batch['encoder_input']
        dec_inputs = batch['decoder_input']
        dec_targets = batch['decoder_target']

        optimizer.zero_grad()

        output = model(enc_inputs, dec_inputs)

        output = output.view(-1, output.size(-1))
        dec_targets = dec_targets.view(-1)

        loss = criterion(output, dec_targets)
        loss.backward()
        optimizer.step()

        preds = output.argmax(dim=-1)

        train_loss += loss.detach().item()

        mask = dec_targets != 0
        correct = (preds == dec_targets) & mask

        train_correct += correct.sum().item()
        train_tokens += mask.sum().item()

    train_loss /= len(train_loader)
    train_acc = train_correct / train_tokens

    train_losses.append(train_loss)
    train_accs.append(train_acc)

    # validation
    model.eval()
    val_loss, val_correct, val_tokens = 0, 0, 0

    with torch.no_grad():
        for batch in val_loader:

            enc_inputs = batch['encoder_input']
            dec_inputs = batch['decoder_input']
            dec_targets = batch['decoder_target']

            output = model(enc_inputs, dec_inputs)

            output = output.view(-1, output.size(-1))
            dec_targets = dec_targets.view(-1)

            loss = criterion(output, dec_targets)

            preds = output.argmax(dim=-1)

            val_loss += loss.detach().item()

            mask = dec_targets != 0
            correct = (preds == dec_targets) & mask

            val_correct += correct.sum().item()
            val_tokens += mask.sum().item()

    val_loss /= len(val_loader)
    val_acc = val_correct / val_tokens

    val_losses.append(val_loss)
    val_accs.append(val_acc)

    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1}/{epochs} TrainLoss={train_loss:.4f} TrainAcc={train_acc:.4f} ValLoss={val_loss:.4f} ValAcc={val_acc:.4f}')

KeyboardInterrupt: 

In [ ]:
# 모델 저장
torch.save(model.state_dict(), 'seq2seq_chatbot.pth')

# 모델 불러오기
model.load_state_dict(torch.load('seq2seq_chatbot.pth'))

